<a name="top"></a><img src="../images/chisel_1024.png" alt="Chisel logo" style="width:480px;" />

# 模块 3.2：生成器：集合
**上一节：[生成器：参数](3.1_parameters.ipynb)**<br>
**下一节：[插曲：Chisel 标准库](3.2_interlude.ipynb)**

## 动机
生成器经常需要处理可变数量的对象，无论是 IO、模块还是测试向量。
集合是应对这类情况的重要构建模块。
本模块将介绍 Scala 集合以及如何在 Chisel 生成器中使用它们。

## 环境设置

In [ ]:
val path = System.getProperty("user.dir") + "/../source/load-ivy.sc"
interp.load.module(ammonite.ops.Path(java.nio.file.FileSystems.getDefault().getPath(path)))

注意我们在这里添加了一个新的导入，因为 `mutable.ArrayBuffer` 位于 `scala.collections` 中。

In [ ]:
import chisel3._
import chisel3.util._
import chisel3.tester._
import chisel3.tester.RawTester.test
import scala.collection._

---
# 生成器与集合<a name="generators-and-collections"></a> 
在本节中，我们将重点讨论*生成器*的概念以及如何使用 Scala 集合作为工具来实现它们。
我们不再将 Chisel 代码视为电路的一个*实例*（即特定电路的描述），
而是将其视为一个电路生成器。

我们将从之前练习中的 FIR 滤波器开始讨论。

In [ ]:
class My4ElementFir(b0: Int, b1: Int, b2: Int, b3: Int) extends Module {
  val io = IO(new Bundle {
    val in = Input(UInt(8.W))
    val out = Output(UInt(8.W))
  })

  val x_n1 = RegNext(io.in, 0.U)
  val x_n2 = RegNext(x_n1, 0.U)
  val x_n3 = RegNext(x_n2, 0.U)
  io.out := io.in * b0.U(8.W) + x_n1 * b1.U(8.W) +
    x_n2 * b2.U(8.W) + x_n3 * b3.U(8.W)
}

这个电路是生成器的一个简单例子，因为它可以生成具有不同系数的 4 抽头滤波器版本。但如果我们想要电路有更多抽头呢？我们将分几个步骤来实现。

- 构建一个抽头可配置 FIR 的软件*黄金模型*。
- 重新设计我们的测试以使用此模型，并确认其工作正常。
- 重构我们的 My4ElementFir，使其支持可配置数量的抽头。
- 使用新的测试平台测试新电路。

<span style="color:blue">**示例：FIR 黄金模型**</span><br><a name="fir-golden-model"></a> 
下面是一个 FIR 电路的 Scala 软件实现。

In [ ]:
/**
  * A naive implementation of an FIR filter with an arbitrary number of taps.
  */
class ScalaFirFilter(taps: Seq[Int]) {
  var pseudoRegisters = List.fill(taps.length)(0)

  def poke(value: Int): Int = {
    pseudoRegisters = value :: pseudoRegisters.take(taps.length - 1)
    var accumulator = 0
    for(i <- taps.indices) {
      accumulator += taps(i) * pseudoRegisters(i)
    }
    accumulator
  }
}

### Seq
注意 `taps` 变成了 `Seq[Int]`，这意味着类的用户可以在构造时传入任意长度的 `Int` 序列。
### 寄存器
通过 `var pseudoRegisters = List.fill(taps.length)(0)` 我们创建了一个 `List`，用于保存之前周期的值。选择 `List` 是因为其将元素添加到头部和移除最后一个元素的语法非常简单。Scala 集合家族中的几乎任何成员都可以使用。我们还将此列表初始化为全零。
### Poke
我们的类添加了一个 poke 函数/方法，模拟将新输入放入滤波器并触发时钟。
### 更新寄存器
代码行 `pseudoRegisters = value :: pseudoRegisters.take(taps.length - 1)` 首先使用 List 的 `take` 方法保留列表中除最后一个元素之外的所有元素，然后使用 `::` 列表拼接运算符将 `value` 添加到缩减后列表的头部。
### 计算输出
一个简单的带累加器的 for 循环，将列表的每个元素乘以其对应的抽头系数。仅包含 `accumulator` 的行将该值作为函数结果返回。
## 调整我们之前的测试以测试黄金模型
现在我们将使用之前的工作来确认我们的黄金模型是有效的。通过一点编辑魔法，我们之前的测试平台转变成了...

In [ ]:
val filter = new ScalaFirFilter(Seq(1, 1, 1, 1))

var out = 0

out = filter.poke(1)
println(s"out = $out")
assert(out == 1)  // 1, 0, 0, 0

out = filter.poke(4)
assert(out == 5)  // 4, 1, 0, 0
println(s"out = $out")

out = filter.poke(3)
assert(out == 8)  // 3, 4, 1, 0
println(s"out = $out")

out = filter.poke(2)
assert(out == 10)  // 2, 3, 4, 1
println(s"out = $out")

out = filter.poke(7)
assert(out == 16)  // 7, 2, 3, 4
println(s"out = $out")

out = filter.poke(0)
assert(out == 12)  // 0, 7, 2, 3
println(s"out = $out")

执行上面的代码块展示了我们的软件模型返回与 My4ElementFir 相同的结果。

## 使用黄金模型测试电路。<a name="use-golden-model-as-test"></a> 
现在我们对黄金模型有了相当的信心，我们重写测试，将电路输出与黄金模型的输出进行比较，而不是使用费力手工制作的示例。下面是一个快速的初版实现。

In [ ]:
val goldenModel = new ScalaFirFilter(Seq(1, 1, 1, 1))

test(new My4ElementFir(1, 1, 1, 1)) { c =>
    for(i <- 0 until 100) {
        val input = scala.util.Random.nextInt(8)

        val goldenModelResult = goldenModel.poke(input)

        c.io.in.poke(input.U)

        c.io.out.expect(goldenModelResult.U, s"i $i, input $input, gm $goldenModelResult, ${c.io.out.peek().litValue}")

        c.clock.step(1)
    }

}

我们的测试运行 100 个周期，并检查两种不同的方法（硬件和软件）在每个步骤中是否同步。

### 需要注意的事项
（即我们在编写时的实际错误。）

1. 将 step 放在正确的位置。软件和硬件的执行方式不同；很容易搞错。
2. 这个测试很脆弱，因为它对 IO 和寄存器的位宽设置非常敏感。实现一个能观察任意数据位宽下行为溢出的软件黄金模型可能很复杂。这里我们只是确保只传入适合的值。

<span style="color:blue">**示例：参数化 FIR 生成器**</span><br><a name="fir-golden-model"></a> 
下面我们创建了一个新的 Filter 类 `MyManyElementsFilter`，它接受一个 `Seq` 常量作为抽头使用。这个列表可以是任意数量的元素。为了稳妥起见，我们还添加了一个 `bitWidth` 参数，允许我们控制电路能处理的数值大小。为了应对可变长度，我们不得不重构寄存器的创建及其连接方式。下面使用的方法只用了可用集合库函数中的一个简单子集。后面的章节将展示如何更简洁地表达行为，同时使发生的事情更清晰。

In [ ]:
class MyManyElementFir(consts: Seq[Int], bitWidth: Int) extends Module {
  val io = IO(new Bundle {
    val in = Input(UInt(bitWidth.W))
    val out = Output(UInt(bitWidth.W))
  })

  val regs = mutable.ArrayBuffer[UInt]()
  for(i <- 0 until consts.length) {
      if(i == 0) regs += io.in
      else       regs += RegNext(regs(i - 1), 0.U)
  }
  
  val muls = mutable.ArrayBuffer[UInt]()
  for(i <- 0 until consts.length) {
      muls += regs(i) * consts(i).U
  }

  val scan = mutable.ArrayBuffer[UInt]()
  for(i <- 0 until consts.length) {
      if(i == 0) scan += muls(i)
      else scan += muls(i) + scan(i - 1)
  }

  io.out := scan.last
}

#### 我们是如何做到的
从第 7、13 和 18 行开始有三个并行的部分。我们使用了一种名为 `ArrayBuffer` 的 Scala 集合类型。`ArrayBuffer` 允许你使用 `+=` 运算符追加元素（也可以插入和删除，但我们不需要）。首先，我们创建一个元素为 `UInt` 的 ArrayBuffer `regs`。然后迭代抽头，将输入添加为第一个元素，然后使用 RegNext 创建寄存器，它将寄存器的输入连接到前一个元素（`regs(i-1)`），并初始化为无符号零（`0.U`）。这些寄存器将根据需要保存输入的先前值。

接下来，我们创建另一个 `UInt` 的 ArrayBuffer `muls`。muls 的每个元素将是一个节点，其第 i 个元素是 `regs(i)` 和 `const(i)` 的乘积。

注意使用了 `scan.last` 方法。它获取集合的最后一个元素，是在 `regs` 构造期间使用的 `regs(i - 1)` 的一个更优雅的替代方案。

### 它是否与 `My4ElementFir` 行为相同？
对我们新版本的一个好的初次测试是看它能否通过我们刚刚应用于 `My4ElementFir` 的测试。我们创建一个 `MyManyElementFir` 实例，并通过更多的数据。

In [ ]:
val goldenModel = new ScalaFirFilter(Seq(1, 1, 1, 1))

test(new MyManyElementFir(Seq(1, 1, 1, 1), 8)) { c =>
    for(i <- 0 until 100) {
      val input = scala.util.Random.nextInt(8)

      val goldenModelResult = goldenModel.poke(input)

      c.io.in.poke(input.U)

      c.io.out.expect(goldenModelResult.U, s"i $i, input $input, gm $goldenModelResult, ${c.io.out.peek().litValue}")

      c.clock.step(1)
    }
}

### 现在让我们测试一批不同大小的 FIR 滤波器
我们创建一些辅助函数：`r` 用于获取随机数；`runOneTest` 用于为特定的抽头集合创建黄金模型和硬件仿真，然后运行至少两倍于抽头数量的数据通过滤波器。

In [ ]:
/** a convenience method to get a random integer
  */
def r(): Int = {
  scala.util.Random.nextInt(1024)
}

/**
  * run a test comparing software and hardware filters
  * run for at least twice as many samples as taps
  */
def runOneTest(taps: Seq[Int]) {
    val goldenModel = new ScalaFirFilter(taps)

    test(new MyManyElementFir(taps, 32)) { c =>
        for(i <- 0 until 2 * taps.length) {
            val input = r()

            val goldenModelResult = goldenModel.poke(input)

            c.io.in.poke(input.U)

            c.io.out.expect(goldenModelResult.U, s"i $i, input $input, gm $goldenModelResult, ${c.io.out.peek().litValue}")

            c.clock.step(1)
        }
    }
}

for(tapSize <- 2 until 100 by 10) {
    val taps = Seq.fill(tapSize)(r())  // create a sequence of random coefficients

    runOneTest(taps)
}

### 只是为了好玩，让我们做一个更大的
以下将在 500 抽头的 FIR 滤波器上运行单个测试。运行可能需要一分钟左右。
（提示：当执行完成时，注意工具栏上的 Scala ● 会变为 Scala ○。）

In [ ]:
runOneTest(Seq.fill(500)(r()))

---
# 硬件集合

<span style="color:blue">**示例：为我们的 FIR 添加运行时可配置的抽头**</span><br>
下面的代码在 FIR 生成器的 IO 中添加了一个额外的 `consts` 向量，允许在电路生成后从外部更改系数。这是通过 Chisel 集合类型 `Vec` 实现的。`Vec` 支持许多 Scala 集合方法，但它只能包含 Chisel 硬件元素。`Vec` 只应在普通 Scala 集合无法工作的情况下使用。基本上，这适用于以下两种情况之一。
1. 你需要在 Bundle 中有一个元素集合，通常是用于 IO 的 Bundle。
2. 你需要通过作为硬件一部分的索引来访问集合（想想寄存器文件）。

In [ ]:
class MyManyDynamicElementVecFir(length: Int) extends Module {
  val io = IO(new Bundle {
    val in = Input(UInt(8.W))
    val out = Output(UInt(8.W))
    val consts = Input(Vec(length, UInt(8.W)))
  })

  // Reference solution
  val regs = RegInit(VecInit(Seq.fill(length - 1)(0.U(8.W))))
  for(i <- 0 until length - 1) {
      if(i == 0) regs(i) := io.in
      else       regs(i) := regs(i - 1)
  }
  
  val muls = Wire(Vec(length, UInt(8.W)))
  for(i <- 0 until length) {
      if(i == 0) muls(i) := io.in * io.consts(i)
      else       muls(i) := regs(i - 1) * io.consts(i)
  }

  val scan = Wire(Vec(length, UInt(8.W)))
  for(i <- 0 until length) {
      if(i == 0) scan(i) := muls(i)
      else scan(i) := muls(i) + scan(i - 1)
  }

  io.out := scan(length - 1)
}

In [ ]:
val goldenModel = new ScalaFirFilter(Seq(1, 1, 1, 1))

test(new MyManyDynamicElementVecFir(4)) { c =>
    c.io.consts(0).poke(1.U)
    c.io.consts(1).poke(1.U)
    c.io.consts(2).poke(1.U)
    c.io.consts(3).poke(1.U)
    for(i <- 0 until 100) {
        val input = scala.util.Random.nextInt(8)

        val goldenModelResult = goldenModel.poke(input)

        c.io.in.poke(input.U)

        c.io.out.expect(goldenModelResult.U, s"i $i, input $input, gm $goldenModelResult, ${c.io.out.peek().litValue}")

        c.clock.step(1)
    }
}


<span style="color:red">**练习：32 位 RISC-V 处理器**</span><br><a name="fir-golden-model"></a>

[寄存器文件](https://en.wikipedia.org/wiki/Register_file) 是构建处理器的重要基础模块。寄存器文件是一个寄存器数组，可以通过多个读或写端口进行读取或写入。每个端口由地址和数据字段组成。

[RISC-V 指令集架构](https://riscv.org/specifications/) 定义了多种变体，其中最简单的一种称为 RV32I。
RV32I 有一个大小为 32 的 32 位寄存器数组。
**索引为 0 的寄存器（第一个寄存器）在读取时始终为零，无论你向它写入了什么**（有一个值为 0 的寄存器常常很有用）。

实现一个用于 RV32I 的寄存器文件，具有一个写端口和参数化数量的读端口。
只有当 `wen`（写使能）被断言时才执行写入。

In [ ]:
class RegisterFile(readPorts: Int) extends Module {
    require(readPorts >= 0)
    val io = IO(new Bundle {
        val wen   = Input(Bool())
        val waddr = Input(UInt(5.W))
        val wdata = Input(UInt(32.W))
        val raddr = Input(Vec(readPorts, UInt(5.W)))
        val rdata = Output(Vec(readPorts, UInt(32.W)))
    })
    
    // A Register of a vector of UInts
    val reg = RegInit(VecInit(Seq.fill(32)(0.U(32.W))))
    
    ???

    
}

In [ ]:
test(new RegisterFile(2) ) { c =>
  def readExpect(addr: Int, value: Int, port: Int = 0): Unit = {
    c.io.raddr(port).poke(addr.U)
    c.io.rdata(port).expect(value.U)
  }
  def write(addr: Int, value: Int): Unit = {
    c.io.wen.poke(true.B)
    c.io.wdata.poke(value.U)
    c.io.waddr.poke(addr.U)
    c.clock.step(1)
    c.io.wen.poke(false.B)
  }
  // everything should be 0 on init
  for (i <- 0 until 32) {
    readExpect(i, 0, port = 0)
    readExpect(i, 0, port = 1)
  }

  // write 5 * addr + 3
  for (i <- 0 until 32) {
    write(i, 5 * i + 3)
  }

  // check that the writes worked
  for (i <- 0 until 32) {
    readExpect(i, if (i == 0) 0 else 5 * i + 3, port = i % 2)
  }
}

<div id="container"><section id="accordion"><div>
<input type="checkbox" id="check-1" />
<label for="check-1"><strong>解答</strong></label>
<article>
<pre style="background-color:#f7f7f7">
    when (io.wen) {
        reg(io.waddr) := io.wdata
    }
    for (i &lt;- 0 until readPorts) {
        when (io.raddr(i) === 0.U) {
            io.rdata(i) := 0.U
        } .otherwise {
            io.rdata(i) := reg(io.raddr(i))
        }
    }

</pre></article></div></section></div>

---
# 你已完成！

[返回顶部](#top)